# 예제 04. 기본 학습 루프
빅데이터프로그래밍 · 5주차

## 목표
- 학습 루프의 다섯 단계를 코드로 쓴다
- 손실이 내려가는 것을 확인한다
- 검증 손실을 함께 기록한다

`예측 → 손실 계산 → 기울기 초기화 → 역전파 → 파라미터 수정`


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. 데이터 준비
정답이 있는 간단한 회귀 문제를 만듭니다. `y = 3x₁ - 2x₂ + 1` 에 약간의 잡음.


In [ ]:
n = 300
X = torch.randn(n, 2)
true_w = torch.tensor([[3.0], [-2.0]])
y = X @ true_w + 1.0 + 0.1 * torch.randn(n, 1)

n_train = 240
train_ds = TensorDataset(X[:n_train], y[:n_train])
val_ds = TensorDataset(X[n_train:], y[n_train:])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

print("학습:", len(train_ds), "검증:", len(val_ds), "batch 수:", len(train_loader))


## 2. 모델 · 손실 · 옵티마이저
선형 모델 하나입니다. 6주차에 층을 늘립니다.


In [ ]:
model = nn.Linear(2, 1).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

print(model)
print("학습할 파라미터:", [tuple(p.shape) for p in model.parameters()])


## 3. 학습 루프 — 다섯 단계

```
for epoch:
    for batch:
        pred = model(x)            1. 예측
        loss = loss_fn(pred, y)    2. 손실 계산
        optimizer.zero_grad()      3. 기울기 초기화
        loss.backward()            4. 역전파
        optimizer.step()           5. 파라미터 수정
```

3번을 빼면 4주차에서 본 기울기 누적 문제가 생깁니다.


In [ ]:
epochs = 30
history = []

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)

        pred = model(bx)                 # 1. 예측
        loss = loss_fn(pred, by)         # 2. 손실

        optimizer.zero_grad()            # 3. 초기화
        loss.backward()                  # 4. 역전파
        optimizer.step()                 # 5. 수정

        train_loss += loss.item() * bx.shape[0]

    train_loss /= len(train_ds)

    # 검증 — 미분 없이
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            val_loss += loss_fn(model(bx), by).item() * bx.shape[0]
    val_loss /= len(val_ds)

    history.append((train_loss, val_loss))
    if epoch % 5 == 0 or epoch == epochs - 1:
        print(f"epoch {epoch:3d}  train {train_loss:.4f}  val {val_loss:.4f}")


## 4. 학습된 파라미터 확인
정답은 3, -2, 편향 1 이었습니다.


In [ ]:
print("학습된 weight:", model.weight.data.cpu().numpy().round(3))
print("학습된 bias  :", model.bias.data.cpu().numpy().round(3))
print("실제 값     : [[ 3. -2.]] / [1.]")


## 5. 손실 곡선


In [ ]:
import matplotlib.pyplot as plt

tr = [h[0] for h in history]
va = [h[1] for h in history]

plt.plot(tr, label="train")
plt.plot(va, label="val")
plt.xlabel("epoch"); plt.ylabel("MSE loss")
plt.legend(); plt.title("loss curve")
plt.show()


## 6. zero_grad 를 빼면 어떻게 되는가
직접 확인해 봅니다.


In [ ]:
bad_model = nn.Linear(2, 1).to(device)
bad_opt = torch.optim.SGD(bad_model.parameters(), lr=0.05)

for epoch in range(5):
    total = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        loss = loss_fn(bad_model(bx), by)
        # bad_opt.zero_grad()    ← 일부러 빼 둠
        loss.backward()
        bad_opt.step()
        total += loss.item()
    print(f"epoch {epoch}  loss {total/len(train_loader):.4f}")


## 직접 해보기
1. 학습률을 0.5로 올리면 손실은 어떻게 되나요?
2. batch size를 8과 128로 바꿔 손실 곡선을 비교하세요.
3. `torch.optim.SGD` 를 `torch.optim.Adam` 으로 바꿔 보세요.


In [ ]:
# 여기에 작성하세요
